# Frozen RL + local search (V9)

Reuse the selected V8 MLP, Set and GNN models. **No training.** Compare their starting plans against heuristic and random starts under the same local-search procedure. Retain the stronger heuristic portfolio as a separate practical baseline.

One trajectory records time budgets and candidate budgets; separate failure traces evaluate the selected plans. See `docs/PILOT_V9.md`. Earlier V8 workflow: `PlacementPilotV8.ipynb`.


In [ ]:
from pathlib import Path
import os, sys, json
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir()
sys.path.insert(0, str(ROOT))
from learnedplacement import anytime_runner as anytime
from learnedplacement.runner import read_json
# Install once if necessary: python -m pip install -e '.[notebook,architecture]'


## Settings and model check

Use your existing completed V8 tuning folder (or V8 models folder). Analysis ZIPs do not contain weights. All available seeds of the direct-selection checkpoints are retained; no new checkpoint is selected using these results.

Defaults: 24 balanced fresh cases, six sizes (6–96 available machines), all six profiles, two search seeds, 32 planning traces and 256 independent evaluation traces. Set `case_mode="full"` **before starting** for all 144 combinations. The balanced pilot cannot isolate every size-by-profile interaction.

Four CPU workers run independent cases with one Torch thread each. Inference is on CPU; this experiment primarily spends time in CPU simulation. No mandatory overall time limit. Keep worker count and settings fixed when resuming. The search may continue beyond 25 seconds to reach 64 unique candidate evaluations, but only plans ready by each cutoff enter the time curves.


In [ ]:
WORKERS = min(4, os.cpu_count() or 1)
SOURCE = ROOT / "results" / "pilot_v8_tuning_w4"  # Existing weights; independent of WORKERS
SETTINGS = read_json(ROOT / "configs" / "pilot_v9.json")
SETTINGS["case_mode"] = "balanced"  # "full" for 144 cases
SETTINGS["seconds"] = [1, 2, 5, 10, 25]
SETTINGS["candidates"] = [8, 16, 32, 64]
SETTINGS["search_seeds"] = [101, 202]
OUTPUT = ROOT / "results" / f"pilot_v9_{SETTINGS['case_mode']}_w{WORKERS}"
CONFIG, BUNDLE, SPECS = anytime.configuration(SOURCE, SETTINGS)
print("Frozen models:", list(SPECS))
print("Cases:", len(anytime.scenarios(CONFIG)))
print("Search trajectories:", len(anytime.scenarios(CONFIG)) * (3 + len(SPECS)) * len(SETTINGS["search_seeds"]))
print("Planning/evaluation traces:", CONFIG["planning_traces"], CONFIG["evaluation_traces"])
print("Output:", OUTPUT)


## Run / resume

This runs the simulator, not real distributed training. Completed searches are saved individually; held-out evaluation saves every few traces. An interrupted search restarts from its beginning so timing is not distorted by loading a warm cache. Completed searches are not repeated. An interrupt cancels queued cases and signals active workers to stop between simulator traces. A running trace may need to finish; do not start another run while workers remain active.


In [ ]:
result = anytime.run(source=SOURCE, output=OUTPUT, settings=SETTINGS, workers=WORKERS)
print(json.dumps(result, indent=2))


## Compare quality at matched budgets

Lower is better. Each curve averages search seeds and model seeds within a workload, then averages workloads. Only cases with all requested methods/seeds available at a budget are included; coverage is printed. No error bars or bottom captions. The objective includes completion time and cost/deadline/noncompletion penalties. Planning time is excluded from that objective.

Time curves measure total warm planning latency, including pool construction, encoding and inference. Count curves compare simulator evaluation efficiency; they do not charge inference/neighborhood costs. Evaluation curves can worsen as planning improves because failure traces are independent.


In [ ]:
print(anytime.status(OUTPUT))
PROFILE = None  # Or "reference", "slow_io", "slow_network", "variable_repair", "high_latency", "slow_network_high_latency"
SIZE = None  # Or 6, 12, 24, 32, 64, 96 available machines
summary = anytime.plot_results(OUTPUT, axis="seconds", profile=PROFILE, size=SIZE)
display(summary)


In [ ]:
display(anytime.plot_results(OUTPUT, axis="candidates", profile=PROFILE, size=SIZE))


## Compact analysis export

Share this ZIP. It includes selected plans, evaluation summaries, search progress, model hashes and checkpoint provenance. Raw failure observations and weights remain local.


In [ ]:
print(anytime.export(OUTPUT))
